# Grammar Scorer

Zero-shot pipeline: T5 grammar correction + LanguageTool spelling detection.

**Inputs:**
- `accepted_rejected.csv` — your labeled caption dataset

**Outputs saved to `/kaggle/working`:**
- `grammar_scores.csv` — per-caption scores, corrected text, user comment, raw errors
- `grammar_t5_model/` — T5 model + tokenizer exported via `save_pretrained()`
- `grammar_artefacts.pkl` — threshold, whitelist, config

## 1. Installs

In [1]:
!pip install transformers torch scikit-learn tqdm --quiet
!pip install language_tool_python --quiet
!pip install nltk --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.0 MB/s eta 0:00:00


True

## 2. Imports & device

In [2]:
import gc
import re
import math
import json
import difflib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, accuracy_score,
    confusion_matrix, classification_report,
)
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import language_tool_python
import warnings
warnings.filterwarnings('ignore')

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/kaggle/working'
print('Using device:', device)


Using device: cuda


## 3. Configuration

In [3]:
GRAMMAR_CONFIG = {
    'correction_model':   'vennify/t5-base-grammar-correction',
    'max_input_tokens':   100,
    'max_new_tokens':     160,
    'alpha':              0.6,    # T5 similarity weight; (1-alpha) = spelling weight
    'whitelist_min_freq': 5,      # tokens flagged in >= N captions → whitelist
}

THRESHOLD_SPLIT = 0.15
THRESHOLD_BETA  = 1.0   # F1 for grammar
SEED            = 42
import numpy as np, torch
torch.manual_seed(SEED)
np.random.seed(SEED)


## 4. Load & clean dataset

In [4]:
import re
import emoji
import unicodedata

def remove_unicode_font_chars(text):
    # Mathematical Bold capitals A-Z: U+1D400–U+1D419
    # Mathematical Bold smalls a-z:   U+1D41A–U+1D433
    # Mathematical Bold Italic, Script, Fraktur, etc. follow in sequence
    # Fastest approach: re-encode to ASCII, replacing anything unresolvable
    result = []
    for char in text:
        # If NFKC decomposition gives us a plain ASCII letter, use it
        normalized = unicodedata.normalize("NFKC", char)
        if normalized.isascii():
            result.append(normalized)
        else:
            # Try to get the unicode name and extract the base letter
            # e.g. "MATHEMATICAL BOLD CAPITAL A" -> "A"
            try:
                name = unicodedata.name(char, "")
                if "MATHEMATICAL" in name or "BOLD" in name or "ITALIC" in name:
                    # Last word in the name is usually the base letter/digit
                    base = name.split()[-1]
                    if len(base) == 1:
                        result.append(base)
                    # else skip it
                else:
                    result.append(char)
            except Exception:
                result.append(char)
    return "".join(result)


def clean_text(text):
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u2026", "...")
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = text.replace('\ufffd', '')
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'\-:;()/]", ' ', text)
    # Remove apostrophes not flanked by letters on both sides
    # keeps: it's, won't, Philippines'  |  removes: ' ' ' ' isolated artifacts
    text = re.sub(r"(?<![a-zA-Z])'|'(?![a-zA-Z])", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_for_models(text):
    text = str(text)

    # 1. Remove full hashtag tokens FIRST, while # is still present
    text = re.sub(r'#\w+', '', text)

    # 2. Remove lead keywords
    text = re.sub(r'^(LOOK|READ|WATCH|CHECK THIS OUT|LISTEN|ICYMI|PSA)\s*[|:\-]?\s*',
                  '', text, flags=re.IGNORECASE)
    text = clean_text(text)
    return text


def clean_for_rules(text):
    """
    Minimal cleaning for formatting/structure rule-based features.
    Preserves |, #, capitalization patterns, lead keywords.
    """
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = text.replace('\ufffd', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
df = pd.read_csv('/kaggle/input/datasets/mariellemayol/accepted-rejected/accepted_rejected.csv')
df['text']   = df['text'].astype(str).str.strip()
df['status'] = df['status'].str.lower().str.strip()
df['issue']  = df['issue'].fillna('NA').str.lower().str.strip()
df['text'] = df['text'].apply(clean_for_models)

# Deduplicate on normalised text BEFORE any split
_norm = df['text'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
n_before = len(df)
df = df[~_norm.duplicated(keep='first')].reset_index(drop=True)
print(f'Deduplicated: {n_before} → {len(df)} rows ({n_before - len(df)} duplicates removed)')

# Remove entries shorter than 10 characters
before_len = len(df)
df = df[df['text'].str.len() >= 10].reset_index(drop=True)
removed_short = before_len - len(df)
print(f'Short text filter: removed {removed_short} entries with < 10 characters ({before_len} → {len(df)})')

print('Dataset shape:', df.shape)
print('\nStatus distribution:')
print(df['status'].value_counts())
print('\nIssue distribution (rejected only):')
print(df[df['status'] == 'rejected']['issue'].value_counts())
display(df.head())


Deduplicated: 1822 → 1804 rows (18 duplicates removed)
Short text filter: removed 2 entries with < 10 characters (1804 → 1802)
Dataset shape: (1802, 3)

Status distribution:
status
accepted    1396
rejected     406
Name: count, dtype: int64

Issue distribution (rejected only):
issue
grammar        197
tone           105
inclusivity    104
Name: count, dtype: int64


,text,status,issue
0,ADVANCING TRANSPARENT YOUTH GOVERNANCE IN THE ...,accepted,na
1,THE CITY OF SAN JOSE DEL MONTE STRENGTHENS YOU...,accepted,na
2,Y-GOV 2026 TRAINING TO STRENGTHEN THE CAPACITY...,accepted,na
3,WE ARE HIRING! The National Youth Commission i...,accepted,na
4,SK Advisory No. 2026-NP003 : The National Yout...,accepted,na


## 5. Grammar module — T5 + LanguageTool

In [6]:
# Load grammar models

print('Loading T5 grammar correction model...')
_grammar_tokenizer = AutoTokenizer.from_pretrained(
    GRAMMAR_CONFIG['correction_model']
)
_grammar_model = AutoModelForSeq2SeqLM.from_pretrained(
    GRAMMAR_CONFIG['correction_model']
).to(device)
_grammar_model.eval()

print('Loading LanguageTool (en-US)...')
_lang_tool = language_tool_python.LanguageTool('en-US')

print('Grammar models ready.')


# Whitelist builder

def build_spelling_whitelist(texts, min_freq=None):
    """
    Run LanguageTool over all texts and collect tokens flagged as spelling
    errors. Any token appearing in >= min_freq captions is added to the
    whitelist (likely a Tagalog word, proper noun, or domain term rather
    than a genuine typo).

    Returns a set of lowercase token strings.
    """
    if min_freq is None:
        min_freq = GRAMMAR_CONFIG['whitelist_min_freq']

    token_caption_counts = Counter()   # token -> how many captions flagged it
    print(f'Building spelling whitelist over {len(texts)} captions...')

    for text in tqdm(texts, desc='whitelist scan'):
        matches = _lang_tool.check(text)
        seen_in_this = set()
        for m in matches:
            if m.category != 'TYPOS':
                continue
            token = text[m.offset: m.offset + m.error_length].strip()
            if not token:
                continue
            token_lc = token.lower()
            if token_lc not in seen_in_this:
                token_caption_counts[token_lc] += 1
                seen_in_this.add(token_lc)

    whitelist = {tok for tok, cnt in token_caption_counts.items() if cnt >= min_freq}
    print(f'Whitelist built: {len(whitelist)} tokens '
          f'(flagged in >= {min_freq} captions)')
    if whitelist:
        sample = sorted(whitelist)[:20]
        print(f'  Sample: {sample}')
    return whitelist


def _is_filtered_error(match, text, whitelist):
    """
    Returns True if a LanguageTool match should be excluded from scoring
    and comments.

    Filters:
    - Non-spelling / non-grammar categories (WHITESPACE, STYLE, PUNCTUATION, etc.)
    - All-caps tokens  → likely acronym
    - Mid-sentence capitalised token → likely proper noun / name
    - Token in corpus whitelist → likely Tagalog or domain-specific term
    """
    # Only care about spelling (TYPOS) and grammar errors
    if match.category not in ('TYPOS', 'GRAMMAR'):
        return True

    token = text[match.offset: match.offset + match.error_length].strip()
    if not token:
        return True

    # All-caps → acronym
    if token.isupper() and len(token) > 1:
        return True

    # Mid-sentence capital → proper noun (not at start of sentence)
    if match.offset > 0 and token[0].isupper():
        # Check a small window before offset for sentence boundary
        preceding = text[max(0, match.offset - 2): match.offset].strip()
        if preceding and preceding[-1] not in '.!?':
            return True

    # Corpus whitelist
    if token.lower() in whitelist:
        return True

    return False


def _make_user_comment(filtered_errors, text):
    """
    Convert surviving LanguageTool errors into plain-English bullet points
    for the user.

    Format examples:
      • Spelling: 'buyed' — did you mean 'bought'?
      • Grammar: 'She go to' — suggestion: 'She goes to'
      • No issues detected.
    """
    if not filtered_errors:
        return 'No grammar or spelling issues detected.'

    lines = []
    for err in filtered_errors:
        token   = text[err['offset']: err['offset'] + err['length']].strip()
        context = err['context'].strip()
        sugg    = err['suggestions']

        if err['category'] == 'TYPOS':
            if sugg:
                lines.append(
                    f"Spelling: '{token}' may be misspelled — "                    f"did you mean '{sugg[0]}'?"                )
            else:
                lines.append(f"Spelling: '{token}' may be misspelled.")
        else:
            # Grammar error — use the LanguageTool message, simplified
            msg = err['message'].rstrip('.')
            if sugg:
                lines.append(f"Grammar: {msg} — suggestion: '{sugg[0]}'.")
            else:
                lines.append(f"Grammar: {msg}.")

    # Deduplicate while preserving order
    seen = set()
    deduped = []
    for line in lines:
        if line not in seen:
            deduped.append(line)
            seen.add(line)

    return ' | '.join(deduped)


# Chunk splitting & T5 correction

def _split_caption_into_chunks(text, tokenizer, max_tokens):
    """
    Two-level split: paragraphs first, sentence fallback for oversized paragraphs.
    Returns list of (chunk_text, para_break_after, is_sentence).
    """
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    for para_idx, para in enumerate(paragraphs):
        token_count  = len(tokenizer.encode(para, add_special_tokens=False))
        is_last_para = (para_idx == len(paragraphs) - 1)
        if token_count <= max_tokens:
            chunks.append((para, not is_last_para, False))
        else:
            sentences = sent_tokenize(para)
            for s_idx, sent in enumerate(sentences):
                is_last_sent = (s_idx == len(sentences) - 1)
                chunks.append((sent, (is_last_sent and not is_last_para), True))
    return chunks


def _correct_chunk(chunk_text, tokenizer, model, max_input_tokens, max_new_tokens):
    """Run T5 correction on a single chunk. Returns corrected string."""
    prefix = 'gec: '
    inputs = tokenizer(
        prefix + chunk_text,
        return_tensors='pt',
        max_length=max_input_tokens + 10,
        truncation=True,
    ).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def score_grammar(text, whitelist):
    """
    Full grammar scoring pipeline for one caption.

    Returns:
        raw_score      float  — blended T5 similarity + spelling score [0, 1]
        corrected_text str    — full corrected caption
        filtered_errors list  — surviving LanguageTool error dicts (post-filter)
        user_comment   str    — plain-English summary for the user
    """
    cfg     = GRAMMAR_CONFIG
    alpha   = cfg['alpha']
    max_in  = cfg['max_input_tokens']
    max_new = cfg['max_new_tokens']

    # T5 correction
    chunks          = _split_caption_into_chunks(text, _grammar_tokenizer, max_in)
    corrected_parts = []
    similarities    = []

    for chunk_text, para_break_after, _ in chunks:
        corrected = _correct_chunk(chunk_text, _grammar_tokenizer,
                                   _grammar_model, max_in, max_new)
        sim = difflib.SequenceMatcher(None, chunk_text, corrected).ratio()
        similarities.append(sim)
        corrected_parts.append((corrected, para_break_after))

    # Rejoin
    rejoined, buf = [], []
    for corrected, para_break in corrected_parts:
        buf.append(corrected)
        if para_break:
            rejoined.append(' '.join(buf))
            buf = []
    if buf:
        rejoined.append(' '.join(buf))
    corrected_text = '\n\n'.join(rejoined)
    t5_similarity  = float(np.mean(similarities)) if similarities else 1.0

    # LanguageTool errors
    lt_matches = _lang_tool.check(text)
    all_errors = [
        {
            'rule_id':     m.rule_id,
            'message':     m.message,
            'offset':      m.offset,
            'length':      m.error_length,
            'context':     m.context,
            'category':    m.category,
            'suggestions': list(m.replacements[:3]),
        }
        for m in lt_matches
    ]

    # Filter errors for scoring and comments
    filtered_errors = [
        e for e, m in zip(all_errors, lt_matches)
        if not _is_filtered_error(m, text, whitelist)
    ]

    # Spelling score
    spelling_errors = [e for e in filtered_errors if e['category'] == 'TYPOS']
    word_count      = max(len(text.split()), 1)
    # Normalise: each spelling error reduces score proportionally to caption length
    # ×10 scale factor so even 1 error in a short caption has visible impact
    spelling_score  = max(0.0, 1.0 - (len(spelling_errors) / word_count * 10))

    # Blended score
    raw_score    = alpha * t5_similarity + (1 - alpha) * spelling_score
    user_comment = _make_user_comment(filtered_errors, text)

    return raw_score, corrected_text, filtered_errors, user_comment


# Quick smoke test
_dummy_whitelist = set()
_test = "She go to the store yesterday and buyed many things. The NPA and mga members was there."
_raw, _corr, _errs, _comment = score_grammar(_test, _dummy_whitelist)
print(f'Input     : {_test}')
print(f'Corrected : {_corr}')
print(f'Score     : {_raw:.4f}')
print(f'Comment   : {_comment}')
print(f'Errors    : {len(_errs)}')


Loading T5 grammar correction model...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading LanguageTool (en-US)...


Grammar models ready.
Input     : She go to the store yesterday and buyed many things. The NPA and mga members was there.
Corrected : She went to the store yesterday and bought many things. The NPA and mga members were there.
Score     : 0.5393
Comment   : Grammar: The pronoun ‘She’ is usually used with a third-person or a past tense verb — suggestion: 'goes'. | Spelling: 'buyed' may be misspelled — did you mean 'bought'? | Spelling: 'mga' may be misspelled — did you mean 'MGA'?
Errors    : 3


## 6. Build spelling whitelist

In [7]:
# Run once over the full dataset — identifies Tagalog words and domain terms
# that LanguageTool would falsely flag as spelling errors.
SPELLING_WHITELIST = build_spelling_whitelist(df['text'].tolist())


Building spelling whitelist over 1802 captions...


whitelist scan: 100%|██████████| 1802/1802 [09:12<00:00,  3.26it/s]

Whitelist built: 666 tokens (flagged in >= 5 captions)
  Sample: ['- yorp', 'abyip', 'accessibl', 'achievign', 'achievin', 'achievingg', 'actiev', 'activ', 'activee', 'activit', 'acupan', 'additiona', 'additionla', 'advocate,s', 'agusan', 'ako', 'alam', 'albay', 'ambag', 'ambisyon']


## 7. Score all captions

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

# Build grammar dimension df (for threshold selection)
def _build_grammar_df(df):
    accepted       = df[df['status'] == 'accepted'].copy()
    rejected_gram  = df[(df['status'] == 'rejected') & (df['issue'] == 'grammar')].copy()
    rejected_other = df[(df['status'] == 'rejected') & (df['issue'] != 'grammar')].copy()
    accepted['label']       = 1
    rejected_other['label'] = 1
    rejected_gram['label']  = 0
    full = pd.concat([accepted, rejected_other, rejected_gram], ignore_index=True)
    cv_df, thresh_df = train_test_split(
        full, test_size=THRESHOLD_SPLIT, stratify=full['label'], random_state=SEED,
    )
    return cv_df[['text','label']].reset_index(drop=True), \
           thresh_df[['text','label']].reset_index(drop=True)

cv_df, thresh_df = _build_grammar_df(df)
print(f'CV pool: {len(cv_df)}  |  Threshold split: {len(thresh_df)}')

def _batch_score(texts):
    raws, corrs, errs, comments = [], [], [], []
    for txt in tqdm(texts, desc='grammar scoring'):
        r, c, e, cm = score_grammar(txt, SPELLING_WHITELIST)
        raws.append(r); corrs.append(c); errs.append(e); comments.append(cm)
    return np.array(raws), corrs, errs, comments

print('\nScoring CV pool...')
cv_raw, cv_corr, cv_errs, cv_comments = _batch_score(cv_df['text'].tolist())

print('\nScoring threshold split...')
th_raw, th_corr, th_errs, th_comments = _batch_score(thresh_df['text'].tolist())

print('\nScoring full dataset...')
all_raw, all_corr, all_errs, all_comments = _batch_score(df['text'].tolist())
print(f'Mean blended score: {all_raw.mean():.4f}')


CV pool: 1531  |  Threshold split: 271

Scoring CV pool...


grammar scoring: 100%|██████████| 1531/1531 [2:36:53<00:00,  6.15s/it]



Scoring threshold split...


grammar scoring: 100%|██████████| 271/271 [26:02<00:00,  5.77s/it]



Scoring full dataset...


grammar scoring: 100%|██████████| 1802/1802 [3:03:10<00:00,  6.10s/it]

Mean blended score: 0.9485


## 8. Threshold selection

In [9]:
def select_threshold(thresh_df, thresh_probs, beta=THRESHOLD_BETA):
    y_true     = thresh_df['label'].values
    thresholds = np.arange(0.05, 0.96, 0.01)
    best_t, best_score = 0.5, 0.0
    for t in thresholds:
        preds = (thresh_probs >= t).astype(int)
        prec, rec, _, _ = precision_recall_fscore_support(
            y_true, preds, average='binary', zero_division=0,
        )
        denom = (beta**2 * prec + rec)
        fb    = (1 + beta**2) * prec * rec / denom if denom > 0 else 0.0
        if fb > best_score:
            best_score = fb; best_t = t
    return round(float(best_t), 2)

def scale_score(raw_prob, threshold):
    import math
    def _sig(x, lo, hi):
        s = 1 / (1 + math.exp(-10 * (x - 0.5)))
        return lo + (hi - lo) * s
    if raw_prob >= threshold:
        return round(_sig((raw_prob - threshold) / (1.0 - threshold + 1e-8), 70.0, 100.0), 1)
    return round(_sig(raw_prob / (threshold + 1e-8), 0.0, 50.0), 1)

threshold = select_threshold(thresh_df, th_raw)
print(f'Grammar threshold: {threshold}  (pass >= {threshold}, fail < {threshold})')


Grammar threshold: 0.05  (pass >= 0.05, fail < 0.05)


## 9. Save scores & model

In [10]:
import pickle, os

DECISION_MAP = {1: 'pass', 0: 'fail'}
df['grammar_raw_score'] = all_raw.round(4)
df['grammar_score']     = [scale_score(p, threshold) for p in all_raw]
df['grammar_decision']  = [DECISION_MAP[int(p >= threshold)] for p in all_raw]
df['grammar_corrected'] = all_corr
df['grammar_comment']   = all_comments
df['grammar_lt_errors'] = [json.dumps(e) for e in all_errs]

out_cols = ['text','status','issue',
            'grammar_raw_score','grammar_score','grammar_decision',
            'grammar_corrected','grammar_comment','grammar_lt_errors']
df[out_cols].to_csv(f'{SAVE_DIR}/grammar_scores.csv', index=False)
print(f'Saved → {SAVE_DIR}/grammar_scores.csv')
display(df[out_cols][['text','grammar_score','grammar_decision','grammar_comment']].head(5))

# Save T5 model for deployment
grammar_model_dir = f'{SAVE_DIR}/grammar_t5_model'
os.makedirs(grammar_model_dir, exist_ok=True)
_grammar_model.save_pretrained(grammar_model_dir)
_grammar_tokenizer.save_pretrained(grammar_model_dir)
print(f'Saved T5 model → {grammar_model_dir}')

# Save artefacts for Notebook 3
artefacts = {
    'threshold':        threshold,
    'whitelist':        SPELLING_WHITELIST,
    'config':           GRAMMAR_CONFIG,
    'grammar_model_dir': grammar_model_dir,
    'scale_score':      scale_score,
}
with open(f'{SAVE_DIR}/grammar_artefacts.pkl', 'wb') as f:
    pickle.dump(artefacts, f)
print(f'Saved artefacts → {SAVE_DIR}/grammar_artefacts.pkl')

print('\nFiles in SAVE_DIR:')
for fn in sorted(os.listdir(SAVE_DIR)):
    print(f'  {fn:50s}  {os.path.getsize(f"{SAVE_DIR}/{fn}")/1e6:6.1f} MB')


Saved → /kaggle/working/grammar_scores.csv


,text,grammar_score,grammar_decision,grammar_comment
0,ADVANCING TRANSPARENT YOUTH GOVERNANCE IN THE ...,99.6,pass,Spelling: 'Joie' may be misspelled — did you m...
1,THE CITY OF SAN JOSE DEL MONTE STRENGTHENS YOU...,99.6,pass,Spelling: 'writeshop' may be misspelled — did ...
2,Y-GOV 2026 TRAINING TO STRENGTHEN THE CAPACITY...,99.7,pass,Spelling: 'Candaba' may be misspelled — did yo...
3,WE ARE HIRING! The National Youth Commission i...,99.4,pass,No grammar or spelling issues detected.
4,SK Advisory No. 2026-NP003 : The National Yout...,99.3,pass,Spelling: 'rldp' may be misspelled — did you m...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved T5 model → /kaggle/working/grammar_t5_model
Saved artefacts → /kaggle/working/grammar_artefacts.pkl

Files in SAVE_DIR:
  __notebook__.ipynb                                     1.0 MB
  grammar_artefacts.pkl                                  0.0 MB
  grammar_scores.csv                                     5.9 MB
  grammar_t5_model                                       0.0 MB
